# 第19周 · 练习：工具定义、tool_result、判分与失败分类

五道练习，从"一个工具的说明书"一路走到"读一条失败轨迹"：

| 练习 | 内容 | 建议时间 |
| --- | --- | --- |
| E1 | 写工具定义（JSON schema） | 15 分钟 |
| E2 | 解析 `tool_use` 块，构造 `tool_result` | 20 分钟 |
| E3 | 把讲义里的 ReAct 循环搬过来，通过检查 | 15 分钟 |
| E4 | 自己写一个答案规范化判分函数 | 25 分钟 |
| E5 | 读五条注入了故障的轨迹，判断失败类别 | 25 分钟 |

**全部练习都在 `MockLLM` 上完成**（不联网、不花钱、结果确定），即使你配置了 API 密钥也一样；`ck` 检查同样只用 MockLLM。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Anthropic 官方 Python SDK（只有配置了 API 密钥、调用真实 Claude 时才用到；
# 没有密钥时，课程用工具包里的 MockLLM 跑通全部代码）
if importlib.util.find_spec("anthropic") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic>=2.3,<3"], check=True)
print("anthropic SDK 已就绪")

In [ ]:
import json
import re
from zero2researcher import agents
from zero2researcher.checks import week19 as ck
from zero2researcher.checks import report

MODEL = agents.MODEL
THINKING = {"type": "adaptive", "display": "summarized"}   # thinking 块返回可读的推理摘要
items = agents.load_eval()
item_by_id = {it["id"]: it for it in items}
print("评测题", len(items), "道 | 五个失败类别:", "、".join(ck.CATEGORIES))

## E1 · 写工具定义

**E1a** · 不看课件，凭记忆写出计算器的工具定义 `calculator_tool`：`name` 为 `"calculator"`；`description` 说清它做什么、接受什么、返回什么、什么时候该用；`input_schema` 里只有一个必填的字符串参数 `expression`（带说明和例子），并禁止额外参数。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.tool_schema(calculator_tool)

**E1b** · 再写第三个工具 `wiki_lookup` 的定义 `wiki_lookup_tool`：按**确切标题**返回离线维基语料里一篇条目的摘要（找不到时返回错误）。它对应的 Python 函数是 `agents.wiki_lookup(title)`——参数名必须一模一样。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.tool_schema(wiki_lookup_tool)

**追问**（写两三句话）：如果把计算器的 `description` 写成一个词 `"math"`，你预计模型会怎样用错它？举一个具体的错误调用。

（在这里回答）

## E2 · 解析 `tool_use` 块，构造 `tool_result`

下面这一格让 MockLLM 回答一道两步的事实题。看看它返回了哪些块：

In [ ]:
llm = agents.MockLLM()
messages = [{"role": "user", "content": item_by_id["f07"]["question"]}]
response = llm.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                      tools=agents.TOOLS, messages=messages)
print("stop_reason:", response.stop_reason)
print("块的类型:", [block.type for block in response.content])

**E2a** · 写 `tool_uses(response)`：返回 `response.content` 里全部的 `tool_use` 块（保持顺序）。不要假设它们在固定的位置——前面还有 `thinking` 和 `text` 块。

In [ ]:
def tool_uses(response):
    # 在这里写你的代码
    raise NotImplementedError


uses = tool_uses(response)
report("找出全部 tool_use 块", [getattr(b, "name", None) for b in uses] == ["wiki_search", "wiki_search"],
       f"这个回复里有 2 个 wiki_search 调用，你找到的是 {[getattr(b, 'name', b) for b in uses]}",
       "遍历 response.content，按 block.type == 'tool_use' 筛选")
for b in uses:
    print(b.id, b.name, b.input)

**E2b** · 写 `run_tool(block)`：用 `agents.TOOL_FUNCTIONS[block.name](**block.input)` 执行工具，返回 `{"type": "tool_result", "tool_use_id": ..., "content": ...}`；工具出错（不安全的表达式、未知的工具名……）时**不要让异常冒出来**，而是返回带 `"is_error": True` 的结果，错误信息写进 `content`。

In [ ]:
def run_tool(block):
    # 在这里写你的代码
    raise NotImplementedError


ck.tool_result_block(run_tool)

**E2c** · 把这一轮补进对话：先把 `response.content` **原样**追加为 assistant 消息，再把两个工具的结果放进**同一条** user 消息。然后再调用一次模型，看它下一步做什么。

漏了某个 `tool_result`、`tool_use_id` 对不上、或者把文字放在 `tool_result` 前面，MockLLM 会像真实 API 一样返回 400，并告诉你哪里错了。把两个结果拆成两条 user 消息呢？真实 API 会把相邻的 user 消息合并成一轮、不报错，MockLLM 也接受，但会提醒你：这样做会让模型以后更少并行调用。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
response2 = llm.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                       tools=agents.TOOLS, messages=messages)
report("第二轮请求被接受", response2.stop_reason == "tool_use")
print("模型的下一步:", [(b.name, b.input) for b in tool_uses(response2)])

## E3 · 把讲义里的 ReAct 循环搬过来

把讲义里的 `run_agent` 复制到下面（调用 E2b 的 `run_tool`）。约定：`run_agent(question, llm, max_steps=8)` 返回一个 dict，含 `answer`、`llm_calls`、`tool_calls`、`input_tokens`、`output_tokens`、`trace`（每次工具调用一条记录）、`messages`（完整对话）、`stop_reason`，以及 `status`——这次运行怎样结束：`"answered"`（`end_turn`）、`"refused"`（`refusal`）、`"truncated"`（`max_tokens`）或 `"max_steps"`（用完步数）。只有 `end_turn` 时才把文字当答案。

检查会用 MockLLM 跑三道题（顺序调用、并行调用、文字答案），检查对话历史是否原样追加、同一轮的结果是否在同一条消息里，再测三种非正常结束：用完 `max_steps`、拒答、被截断。

In [ ]:
STATUS = {"end_turn": "answered", "refusal": "refused", "max_tokens": "truncated"}


def run_agent(question, llm, max_steps=8):
    # 在这里写你的代码
    raise NotImplementedError


ck.react_loop(run_agent)

## E4 · 自己写一个判分函数

工具包的 `agents.is_correct` 你在讲义里已经用过。现在自己写一个简化版 `my_is_correct(pred, gold)`，规则如下：

1. `pred` 是 `None` 或空字符串 → `False`；
2. 如果 `pred` 里有 `Final answer:`（大小写不敏感），只看**最后一个**标记之后的部分；
3. 去掉数字里的千位分隔逗号（`1,953` → `1953`，但 `21.3, so` 里的逗号不动）；
4. `gold` 是数字（`int` 或 `float`）时：找出所有数字；一个数字都没有 → `False`。有标记时，标记后的数必须**只有一个值**——出现两个不同的数（`1953, 1954` 这种"两头下注"，或 `about 21.3, so 21`）→ `False`，否则取这个数；没有标记时取全文**最后一个**数。与 `gold` 之差不超过 `1e-6` 算对；
5. `gold` 是文字时：两边都转小写、只保留单词、去掉冠词 `a / an / the`，再比较是否相等。

提示：`re.finditer(r"final answer\s*:", text, flags=re.I)` 找标记；`re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", text)` 去千位逗号；`re.findall(r"-?\d+(?:\.\d+)?", text)` 找数字，`len(set(...)) > 1` 判断是否不止一个值；`re.findall(r"\w+", s.lower())` 取单词。检查会把你的函数和 `agents.is_correct` 在 17 个边界用例上逐一比对。

In [ ]:
def my_is_correct(pred, gold):
    # 在这里写你的代码
    raise NotImplementedError


ck.normalized_match(my_is_correct)

**追问**（写两三句话）：17 个用例里，有两个"内容对了、却被判错"的用例（`twenty-one` 与 `about 21.3, so 21 full bags`）。它们在失败分析里属于哪一类？为什么不干脆把规则放宽，让 `twenty-one` 也算对？另外，`Final answer: 1953, 1954` 为什么也要判错？（提示：规则越宽松，越可能把错的判成对的；规则必须在看结果之前定好，并对所有系统一视同仁。）

（在这里回答）

## E5 · 读轨迹，判断失败类别

`ck.failure_case(编号)` 返回一道题和一个**在这道题上注入了某种故障**的 MockLLM（它不会告诉你注入了什么：`faults` 属性只显示 `"custom"`）。用你在 E3 写的 `run_agent` 跑五个案例，**读每一条轨迹**，判断失败属于哪一类（每条只标一个主因）：

| 类别 | 定义 |
| --- | --- |
| 工具选择错误 | 调用了不该用的工具；workflow 里路由走错了分支 |
| 参数错误 | 工具选对了，但传进去的参数错了（算式列错、检索词查不到所需条目） |
| 提前结束 | 还没拿到所需信息（或没做完所需的计算）就给出答案 |
| 幻觉 / 编造 | 观察结果里已有正确信息，答案却与之矛盾，或凭空捏造 |
| 格式不匹配 | 答案内容其实对了，只是判分器没认出来 |

判断顺序（第一条成立的就是主因）：① 内容对、形式错 → 格式不匹配；② 用错了工具 → 工具选择错误；③ 工具对、参数错 → 参数错误；④ 没拿到所需信息就作答 → 提前结束；⑤ 有信息却答错 → 幻觉 / 编造。

In [ ]:
for case in "ABCDE":
    item, faulty_llm = ck.failure_case(case)
    out = run_agent(item["question"], faulty_llm)
    print(f"========== 案例 {case} · 题 {item['id']} · 标准答案 {item['gold']} · "
          f"判分 {agents.grade(item, out['answer'])}")
    agents.print_transcript(out["messages"])
    print()

把你的判断填进 `labels`（类别写中文或英文都可以，例如 `"参数错误"` 或 `"bad args"`），再在下面的表里为每个判断写一句**轨迹里的证据**。

In [ ]:
labels = {}      # 例如 labels["A"] = "参数错误"
# 在这里写你的代码
raise NotImplementedError
for case in "ABCDE":
    ck.failure_category(labels.get(case, ""), case)

| 案例 | 类别 | 轨迹里的证据（引用一句） |
| --- | --- | --- |
| A | | |
| B | | |
| C | | |
| D | | |
| E | | |